# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

optimizer = ModelOptimizer("UserKNN_jaccard")

STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + '_jaccard'

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "jaccard",
        "topK": optuna_trial.suggest_int("topK", 5, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=30
)

[I 2025-11-20 16:28:20,757] Using an existing study with name 'UserKNNCFRecommender_jaccard' instead of creating a new one.


  0%|          | 0/30 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 1672.43 column/sec. Elapsed time 16.20 sec
  Fold 1/5 - Score: 0.21900096535682678
Similarity column 27095 (100.0%), 1269.17 column/sec. Elapsed time 21.35 sec
  Fold 2/5 - Score: 0.21840421855449677
Similarity column 27095 (100.0%), 1164.55 column/sec. Elapsed time 23.27 sec
  Fold 3/5 - Score: 0.21915850043296814
Similarity column 27095 (100.0%), 1517.53 column/sec. Elapsed time 17.85 sec
  Fold 4/5 - Score: 0.2171059101819992
[I 2025-11-20 16:30:14,093] Trial 72 finished with value: 0.21841739118099213 and parameters: {'topK': 224, 'shrink': 0, 'normalize': False, 'feature_weighting': 'none'}. Best is trial 16 with value: 0.22032096982002258.
Similarity column 27095 (100.0%), 1162.11 column/sec. Elapsed time 23.32 sec
  Fold 1/5 - Score: 0.216588094830513
Similarity column 27095 (100.0%), 1579.45 column/sec. Elapsed time 17.15 sec
  Fold 2/5 - Score: 0.21637526154518127
Similarity column 27095 (100.0%), 1509.69 column/sec. Elapsed time 17.95 sec
  F

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [8]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "jaccard",
        "topK": optuna_trial.suggest_int("topK", 360, 400),
        "shrink": optuna_trial.suggest_int("shrink", 0, 20),
        "normalize": True,
        "feature_weighting": "none",
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-20 18:07:36,155] Using an existing study with name 'UserKNNCFRecommender_jaccard_refined' instead of creating a new one.


  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 1729.97 column/sec. Elapsed time 15.66 sec
  Fold 1/5 - Score: 0.2191651463508606
Similarity column 27095 (100.0%), 1661.44 column/sec. Elapsed time 16.31 sec
  Fold 2/5 - Score: 0.21853089332580566
Similarity column 27095 (100.0%), 1669.74 column/sec. Elapsed time 16.23 sec
  Fold 3/5 - Score: 0.22019322216510773
Similarity column 27095 (100.0%), 1621.37 column/sec. Elapsed time 16.71 sec
  Fold 4/5 - Score: 0.21817810833454132
[I 2025-11-20 18:09:15,963] Trial 4 finished with value: 0.21901683509349823 and parameters: {'topK': 399, 'shrink': 18}. Best is trial 2 with value: 0.22020494937896729.
Similarity column 27095 (100.0%), 1625.47 column/sec. Elapsed time 16.67 sec
  Fold 1/5 - Score: 0.21948905289173126
Similarity column 27095 (100.0%), 1678.59 column/sec. Elapsed time 16.14 sec
  Fold 2/5 - Score: 0.2193932831287384
Similarity column 27095 (100.0%), 1579.24 column/sec. Elapsed time 17.16 sec
  Fold 3/5 - Score: 0.2211681753396988
Similarity co

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Trial 1:
Best Value: 0.22085218131542206
Best Params: {'topK': 385, 'shrink': 1, 'normalize': True, 'feature_weighting': 'none'}

- Trial 2:
Best Value: 0.2208811491727829
Best Params: {'topK': 367, 'shrink': 0}

Best Value: 0.2208811491727829

Best Params: {'topK': 367, 'shrink': 0, 'normalize': True, 'feature_weighting': 'none'}